# The four labels as a time series: when to send a frame to the VLM

Every CSV row is one frame pair, so each video is already a **label time series**: sort its
rows by frame and you get TRANSITION / PURSUIT / REFIXATION / STABLE, one per second. No
download, no DINOv2, no training: everything here reads files already on Drive.

**The idea under test:** don't send every changing frame. While the scene is moving, wait;
send **one frame when the view settles**.

```
labels:   S  S  T  T  T  S  S  R  S
send A:         ^  ^  ^        ^         every non-STABLE frame (the design PDF's gate)
send C:                  ^               first settled frame after the moving stretch
```

| state | labels | meaning |
|---|---|---|
| **moving** | TRANSITION, PURSUIT | frame similarity low: the head/scene is changing |
| **settled** | REFIXATION, STABLE | frame similarity high: the scene holds still |

| policy | sends frame t when |
|---|---|
| **A** all but STABLE | label is not STABLE (current gate) |
| **B** TRANSITION + REFIX | label is TRANSITION or REFIXATION (discard PURSUIT + STABLE) |
| **C** settle | first settled label after a moving stretch |
| **D** settle + refix | C, plus every REFIXATION inside a settled stretch (new gaze target) |
| **E** D + timeout | D, plus one frame every 5 s during a long moving stretch (walking) |
| **F** E + hysteresis | E with P(moving) thresholds 0.6 in / 0.4 out (models with probabilities) |
| **G** E, settle ≥ 2 s | E, but a settle only counts once it has lasted 2 s (ignores one-second blips; 1 s more delay) |
| **U** every N s | **uniform sampling, no sensors**: one frame every 1, 2, 3, 4, 5, 6, 8, 10, 12, 15 s |

**U is the baseline that matters.** A gaze/IMU policy is only worth its sensors if it beats
uniform sampling **at the same number of sends per minute**. Section 5 measures exactly that.

## Questions, in order

1. **How long are the runs?** If TRANSITION runs are mostly 1 frame, C saves little over A.
   If they are often 3+ frames, it saves a lot.
2. **What ends a moving run:** STABLE, REFIXATION, or the end of the video?
3. **Replay the policies on the ground-truth labels:** the best each policy could do.
4. **Replay them on each model's predictions** for the 12 test videos, and compare every
   policy with uniform sampling at equal compute.
5. **Look at one video as a timeline.**

## Metrics (always judged against the DINOv2 labels)

| metric | meaning |
|---|---|
| sends / min | compute: frames sent per minute of video |
| settles caught % | true moving → settled boundaries that got a sent frame within ±1 s |
| send offset (s) | when caught: −1 = the frame before the settle, 0 = at it, +1 = one late |
| refix caught % | true REFIXATIONs inside a settled stretch that got a send at or 1 s after |
| mid-motion sends % | sends in the middle of a moving stretch (likely blurred, not attended) |
| **stale settled %** | on frames where the wearer has **settled** (true REFIXATION/STABLE), how often the CLS cosine to the **last sent** frame is below τ_frame: the VLM's view is out of date while the person is looking. **The settle policies are judged on this** |
| stale moving % | the same on moving frames: the price of not sending during motion (by design) |
| stale frames % | both together (what the first version of this notebook reported) |

## 1. Clone, mount, paths

In [ ]:
import os, sys, json, subprocess

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'     # 'main' once merged
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('eyehead/policy.py'), f'eyehead/policy.py is not on branch {BRANCH!r}'

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/GazeVLM'
FOLDS      = os.path.join(DRIVE_DIR, 'folds')
THRESH     = os.path.join(DRIVE_DIR, 'thresholds.json')
FEAT_CACHE = os.path.join(DRIVE_DIR, 'feat_cache.npz')        # CLS per frame, from gatetrain
RUNS       = os.path.join(DRIVE_DIR, 'runs')
BASELINE   = os.path.join(DRIVE_DIR, 'release', 'gate_v1', 'test_preds.csv')
OUT        = os.path.join(RUNS, 'policy')
os.makedirs(OUT, exist_ok=True)

for p in (os.path.join(FOLDS, 'train_val_fold1.csv'), os.path.join(FOLDS, 'test.csv'), THRESH):
    assert os.path.exists(p), f'missing {p}'
print('feature cache:', 'found' if os.path.exists(FEAT_CACHE)
      else 'NOT found -- the stale-frames metric will be skipped')
print('old gate preds:', 'found' if os.path.exists(BASELINE) else 'not found')

## 2. Run everything

Reads the labels of all 70 videos (fold file + test file), splits each video into clips of
chained frame pairs, and replays every policy on the ground truth and on each model's test
predictions (`test_preds.csv` from the eyehead notebook). Results are also written to
`MyDrive/GazeVLM/runs/policy/`.

In [ ]:
from eyehead import policy as pol
from eyehead.features import NAMES

PREDS = {'eye only': 'eyehead_eye_only', 'IMU only': 'eyehead_head_only',
         'eye + IMU': 'eyehead_eye_head', 'full': 'eyehead'}
PREDS = {k: os.path.join(RUNS, v, 'test_preds.csv') for k, v in PREDS.items()}
TIMEOUT, HYST, MIN_SETTLE = 5, (0.6, 0.4), 2

o = pol.run_all(FOLDS, THRESH, FEAT_CACHE, PREDS, BASELINE, TIMEOUT, HYST, MIN_SETTLE)
res = o['results']

for fname, key in (('runs.csv', 'runs'), ('run_lengths_all.csv', 'lengths_all'),
                   ('run_lengths_test.csv', 'lengths_test'), ('policy_results.csv', 'results'),
                   ('vs_uniform.csv', 'vs_uniform')):
    o[key].to_csv(os.path.join(OUT, fname), index=False)
o['moving_next_all'].to_csv(os.path.join(OUT, 'moving_run_next.csv'))
o['transitions_all'].to_csv(os.path.join(OUT, 'transitions.csv'))
print('wrote', OUT)

## 3. How long do the labels last?

`pct_runs_ge3` is the number that decides question 1: the share of runs lasting 3 s or
more. `pct_frames_in_runs_ge3` is the share of that label's frames that sit in such runs,
which is roughly what C can save over A.

In [ ]:
print('RUN LENGTHS, all videos (1 frame = 1 s)')
display(o['lengths_all'].round(2))
print('RUN LENGTHS, test videos')
display(o['lengths_test'].round(2))

rt = o['runs']
COL4 = {'TRANSITION': 'tab:blue', 'PURSUIT': 'tab:orange', 'REFIXATION': 'tab:red', 'STABLE': 'tab:green'}
bins = np.arange(1, 17) - 0.5

fig, ax = plt.subplots(1, 3, figsize=(20, 4.6))
for c in NAMES.values():
    L = rt[(rt.kind == 'class') & (rt.run == c)].length.clip(upper=15)
    ax[0].hist(L, bins=bins, histtype='step', lw=1.8, color=COL4[c], density=True, label=c)
ax[0].set_xlabel('run length in s (15 = 15+)'); ax[0].set_ylabel('share of runs')
ax[0].set_title('how long each label lasts'); ax[0].legend(fontsize=8)

for st, col in (('moving', 'tab:purple'), ('settled', 'tab:gray')):
    L = rt[(rt.kind == 'state') & (rt.run == st)].length.clip(upper=15)
    ax[1].hist(L, bins=bins, histtype='step', lw=2, color=col, density=True, label=st)
ax[1].set_xlabel('run length in s (15 = 15+)')
ax[1].set_title('moving {TRANSITION, PURSUIT} vs settled {REFIXATION, STABLE}'); ax[1].legend()

mv = rt[(rt.kind == 'state') & (rt.run == 'moving')]
ct = pd.crosstab(mv['mix'], mv['next'], normalize='index') * 100
order = [c for c in ('STABLE', 'REFIXATION', 'END') if c in ct.columns]
ct[order].plot(kind='bar', stacked=True, ax=ax[2], rot=0,
               color=[{'STABLE': 'tab:green', 'REFIXATION': 'tab:red', 'END': 'tab:gray'}[c] for c in order])
ax[2].set_ylabel('% of moving runs'); ax[2].set_title('what ends a moving run')
plt.tight_layout(); plt.show()

print('what ends a moving run (counts, all videos; rows = what the run was made of):')
display(o['moving_next_all'])
tm = o['transitions_all']
print('label -> next label, % of each row (all videos):')
display((100 * tm.div(tm.sum(1), axis=0)).round(1))

## 4. The policies on the ground-truth labels

This is the **ceiling** for each policy: what it would do with perfect labels. The uniform
rows use the same videos. Read **stale settled %** next to **sends / min**: the settle
policies should keep the settled view fresh while sending far less than A.

In [ ]:
COLS = ['sends / min', 'send %', 'settles caught %', 'send offset (s)', 'refix caught %',
        'mid-motion sends %', 'stale settled %', 'stale moving %', 'stale frames %']
COLS = [c for c in COLS if c in res.columns]
gt = res[res.source.isin(['ground truth, all videos', 'uniform, all videos'])]
display(gt.set_index(['source', 'policy'])[COLS].round(2))

## 5. Every policy against uniform sampling (12 test videos)

Same policies, driven by each model's predicted labels and by the true labels, all on the
test videos. In the plots, the **grey line is uniform sampling** at every rate; a policy is
worth its sensors only if it sits **on the better side of that line**:

- left plot: stale settled % (lower is better)
- right plot: settles caught % (higher is better)

The table below the plots does the same comparison numerically: uniform sampling's value is
interpolated at the policy's own sends/min, and **advantage > 0 means the policy beats
uniform sampling at equal compute**.

In [ ]:
test_rows = res[~res.source.isin(['ground truth, all videos', 'uniform, all videos'])]
display(test_rows[test_rows.source != 'uniform, test']
        .set_index(['source', 'policy'])[COLS].round(2))

SRC_COL = {'ground truth, test': 'k', 'old gate': 'tab:gray', 'eye only': 'tab:orange',
           'IMU only': 'tab:green', 'eye + IMU': 'tab:purple', 'full': 'tab:blue'}
sens = test_rows[test_rows.source != 'uniform, test']
pols = sorted(sens.policy.unique())
MARK = dict(zip(pols, 'osD^vP*X'))
uni = res[res.source == 'uniform, test'].sort_values('sends / min')

panels = [m for m in ('stale settled %', 'settles caught %') if m in res.columns]
fig, ax = plt.subplots(1, len(panels), figsize=(9 * len(panels), 6.5))
for a_, m in zip(np.atleast_1d(ax), panels):
    a_.plot(uni['sends / min'], uni[m], '-', color='0.6', lw=2.5, zorder=1)
    for _, u in uni.iterrows():
        a_.annotate(u.policy.replace('U: every ', ''), (u['sends / min'], u[m]),
                    fontsize=7, color='0.4', xytext=(3, 3), textcoords='offset points')
    for _, r in sens.iterrows():
        a_.scatter(r['sends / min'], r[m], marker=MARK[r.policy], s=80, zorder=3,
                   color=SRC_COL.get(r.source, 'tab:red'), edgecolor='k', linewidth=.4)
    a_.set_xlabel('sends per minute  (compute)'); a_.set_ylabel(m); a_.grid(alpha=.3)
    a_.set_title(f"{m}: {'lower' if 'stale' in m else 'higher'} than the grey line = beats uniform")
h1 = [Line2D([], [], color='0.6', lw=2.5, label='uniform sampling')] + \
     [Line2D([], [], marker='o', ls='', color=c, label=s) for s, c in SRC_COL.items()
      if s in set(sens.source)]
h2 = [Line2D([], [], marker=MARK[p], ls='', color='k', label=p) for p in pols]
a0 = np.atleast_1d(ax)[0]
a0.add_artist(a0.legend(handles=h1, loc='upper right', fontsize=8, title='source'))
np.atleast_1d(ax)[-1].legend(handles=h2, loc='lower right', fontsize=8, title='policy')
plt.tight_layout(); plt.show()

v = o['vs_uniform']
adv = [c for c in v.columns if c.endswith('advantage')]
show = v[v.source != 'ground truth, all videos'].set_index(['source', 'policy'])
print('advantage over uniform sampling at the same sends/min (positive = policy is better):')
display(show[['sends / min'] + adv].round(2)
        .style.background_gradient(cmap='RdYlGn', subset=adv, vmin=-15, vmax=15))

## 6. One test video as a timeline

Top row: the true label of every second. Below: each vertical tick is a frame sent to the
VLM. Change `CLIP` to look at another test video (it defaults to the longest).

In [ ]:
clips = o['clips']
test_ids = [i for i, c in enumerate(clips) if c['is_test']]
CLIP = max(test_ids, key=lambda i: len(clips[i]['labels']))

c = clips[CLIP]
lab, n = c['labels'], len(c['labels'])
SHOW = [('uniform, test', 'U: every 4 s'),
        ('ground truth, test', 'A: all but STABLE'),
        ('ground truth, test', f'E: D + {TIMEOUT}s timeout'),
        ('ground truth, test', f'G: E, settle >= {MIN_SETTLE}s'),
        ('full', 'A: all but STABLE'),
        ('full', f'E: D + {TIMEOUT}s timeout'),
        ('full', f'G: E, settle >= {MIN_SETTLE}s')]
SHOW = [s for s in SHOW if (s[0], s[1], CLIP) in o['sent']]

fig, ax = plt.subplots(len(SHOW) + 1, 1, figsize=(19, 1.0 * len(SHOW) + 1.8), sharex=True)
ax[0].bar(np.arange(1, n + 1), 1, width=1.0, color=[COL4[NAMES[v]] for v in lab])
ax[0].set_yticks([])
ax[0].set_ylabel('true label', rotation=0, ha='right', va='center')
ax[0].legend([plt.Rectangle((0, 0), 1, 1, color=COL4[k]) for k in NAMES.values()],
             list(NAMES.values()), ncol=4, fontsize=8, loc='lower right', bbox_to_anchor=(1, 1.02))
ax[0].set_title('%s   (%d s)' % (c['sequence'], n + 1), loc='left')
for a_, (src, p) in zip(ax[1:], SHOW):
    m = o['sent'][(src, p, CLIP)]
    a_.vlines(np.flatnonzero(m), 0, 1, color='k', lw=1.1)
    a_.set_yticks([])
    a_.set_ylabel('%s\n%s\n%d sends' % (src, p, m.sum()), rotation=0, ha='right', va='center', fontsize=8)
ax[-1].set_xlabel('frame (s)')
plt.tight_layout(); plt.show()

---

## Reading it

| look at | what it tells you |
|---|---|
| `pct_runs_ge3` for TRANSITION and PURSUIT | whether the settle idea can save much at all |
| what ends a moving run | how often the view settles into STABLE vs REFIXATION |
| sends / min, A vs C/D/E/G on ground truth | the compute the settle policy saves, with perfect labels |
| **stale settled %** | whether the VLM's view is current while the wearer is looking; the settle policies' real cost |
| **advantage over uniform** (section 5) | the decisive number: positive means the sensors buy something that periodic sampling at the same rate does not |
| ground-truth vs model rows | how much of the ceiling each model reaches |
| G vs E | whether one-second settled blips are worth a send |
| F vs E | whether hysteresis removes spurious sends from single-frame mispredictions |

## Caveats

- 1 fps: every decision lands on a 1 s grid, and "the view settled" is only known one frame
  after it happens.
- Staleness uses the whole-frame CLS only. A gaze-region change inside an unchanged scene
  (REFIXATION) does not make a frame stale by this measure; refix caught % covers that.
- Uniform sampling here starts at frame 0 of each video; a different phase changes its
  numbers slightly but not its trend.
- The labels are DINOv2 proxies. Whether a VLM answers better with these frames is still
  untested.